# RES-200: Effect of Mouth-Region Resolution on Arabic Lip Reading (LRW-AR)

One notebook with all the code. Every function and model lives in the cells below.

**Family A: small 3D-CNN, trained from scratch:** 3D-CNN baseline, + LSTM, + GRU, + temporal attention, + spatiotemporal attention.
**Family B: 3D ResNet-18, trained from scratch:** 3D ResNet-18, + temporal attention.
**Family C: 2D ResNet per frame, ImageNet-pretrained:** ResNet-18 alone, + TCN, + TCN + attention, + BiGRU + attention;
ResNet-50 alone, + TCN, + temporal attention, + TCN + attention, + BiGRU + attention.

16 models x 6 resolutions (96, 64, 48, 32, 24, 16) x 3 seeds (11, 23, 37) = 288 runs.
Protocol: 30 epochs, batch 16, AdamW (one-cycle, peak LR 1e-3, weight decay 1e-4), mixed precision, gradient clipping 1.0;
the best-validation checkpoint is tested once on the 458 test clips.
(An early ResNet-50 + attention model trained from scratch, `r50_attention`, did not learn and is not used in the paper.)

Steps: 1 setup, 2 dataset, 3 settings, 4 data loader, 5 models, 6 smoke test, 7 training + evaluation, 8 load data,
9 status, 10 pilot, 11 full experiment (resumable), 12-14 tables, figures, download.


## 1. Setup: GPU and folders

In [ ]:
import os, sys, json, time, random, glob, csv
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print('GPU OK:', torch.cuda.get_device_name(0))
else:
    print('*' * 70)
    print('NO GPU. Training on CPU is extremely slow (fine for the smoke test only).')
    print('In Colab: Runtime > Change runtime type > T4 GPU, then run again.')
    print('*' * 70)

IN_COLAB = IN_KAGGLE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    WORK = Path('/content/drive/MyDrive/RES200_lipreading')
    DATA_ROOT = Path('/content/data')
except Exception:   # not Colab (Kaggle also ships google.colab but mount fails)
    if Path('/kaggle/working').exists():       # Kaggle notebook
        IN_KAGGLE = True
        WORK = Path('/kaggle/working/RES200_lipreading')
        DATA_ROOT = Path('/tmp/res200_data')    # big temp disk, not saved
    else:
        WORK = Path('RES200_lipreading').resolve()   # local Jupyter: next to this notebook
        DATA_ROOT = WORK / 'data'

RESULTS = WORK / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle: continue from finished runs attached as an input dataset, if any.
if IN_KAGGLE and not (RESULTS / 'results_per_run.csv').exists():
    import shutil
    found = sorted(Path('/kaggle/input').rglob('results_per_run.csv'),
                   key=lambda f: len(pd.read_csv(f)), reverse=True)
    if found:
        shutil.copytree(found[0].parent, RESULTS, dirs_exist_ok=True)
        print('Copied finished runs from', found[0].parent, '->', len(pd.read_csv(found[0])), 'runs')
    else:
        print('No finished runs found in /kaggle/input (starting from zero).')
print('Results folder:', RESULTS)

## 2. Get the dataset
Downloads the preprocessed 23-word LRW-AR subset (`.npy` mouth-region clips) and finds the folder with `train/ val/ test/`.
Expected: train 3672, val 460, test 458 clips. To use a local copy of the archive instead, set `ARCHIVE_ON_DRIVE` to its path.

In [ ]:
import tarfile, zipfile, subprocess

FILE_ID = '1jOKbfjf4kXUOGlCoDoNaj20Kj9DUjI8t'
ARCHIVE_ON_DRIVE = ''   # optional fallback, e.g. '/content/drive/MyDrive/dataset-lip.tar.gz'


def find_split_root(base):
    for d, subdirs, _ in os.walk(base):
        if {'train', 'val', 'test'} <= set(subdirs):
            return Path(d)
    return None


DATA = find_split_root(DATA_ROOT)
if DATA is None:
    if ARCHIVE_ON_DRIVE:
        archive = Path(ARCHIVE_ON_DRIVE)
    else:
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
            import gdown
        archive = DATA_ROOT.parent / 'dataset_archive'
        gdown.download(id=FILE_ID, output=str(archive), quiet=False)
    with open(archive, 'rb') as f:
        magic = f.read(8)
    if zipfile.is_zipfile(archive):
        zipfile.ZipFile(archive).extractall(DATA_ROOT)
    elif magic.startswith(b'Rar!'):
        subprocess.run(['unrar', 'x', '-o+', '-idq', str(archive), str(DATA_ROOT) + '/'], check=True)
    else:  # tar / tar.gz (the LRW-AR archive is a gzip tar)
        with tarfile.open(archive) as t:
            t.extractall(DATA_ROOT)
    if not ARCHIVE_ON_DRIVE:
        archive.unlink()   # free disk space
    DATA = find_split_root(DATA_ROOT)

assert DATA, 'Could not find a folder with train/ val/ test/. Check the archive contents.'
print('DATA =', DATA)
for s in ['train', 'val', 'test']:
    n = len(glob.glob(f'{DATA}/{s}/*/*.npy'))
    print(f'{s}: {n} clips, {len(os.listdir(DATA / s))} classes')

## 3. Settings
`MODELS_TO_RUN`, `SEEDS` and `RESOLUTIONS` decide which runs the full experiment trains.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 1e-3          # peak LR of the one-cycle schedule
WEIGHT_DECAY = 1e-4
RESOLUTIONS = [96, 64, 48, 32, 24, 16]
SEEDS = [11, 23, 37]
# The 16 models reported in the paper (16 models x 6 resolutions x 3 seeds = 288 runs).
MODELS_TO_RUN = ['3d_cnn', '3d_cnn_lstm', '3d_cnn_gru', '3d_cnn_attention', '3d_cnn_st_attention',
                 'r3d_18', 'r3d_attention',
                 'r18p_avg', 'r18p_tcn', 'r18p_tcn_att', 'r18p_bigru_att',
                 'r50p_avg', 'r50p_tcn', 'r50p_att', 'r50p_tcn_att', 'r50p_bigru_att']

USE_BOTH_GPUS = False    # True uses both GPUs with DataParallel (needs more RAM)

# Stop starting new runs after this many hours, so the session can still save tables and files.
# Finished runs are kept, so a new session continues where the last one stopped.
TIME_BUDGET_HOURS = 10.5
SESSION_START = time.time()

CLASSES = [
    'من', 'في', 'باسم', 'بعد', 'اليوم', 'الدولة', 'الذي', 'تنظيم',
    'الرئيس', 'السعودية', 'علي', 'صالح', 'طائرات', 'شخصا', 'شمال',
    'الموجز', 'عن', 'عدد', 'عليكم', 'الحكومة', 'قناة', 'قطر', 'قتل',
]
NUM_CLASSES = len(CLASSES)

## 4. Data loader
Each clip is standardized to zero mean and unit standard deviation using its own pixel values. Augmentation is used for training only: horizontal flip (p = 0.5) and temporal masking (p = 0.5).

In [ ]:
def normalize_clip(sequence: np.ndarray) -> np.ndarray:
    # Per-clip standardization: zero mean, unit std.
    sequence = sequence.astype(np.float32)
    return (sequence - sequence.mean()) / (sequence.std() + 1e-6)


class PreprocessedLRWARDataset(Dataset):
    # Loads every clip into RAM once (float16), so training is not slowed by disk reads.

    def __init__(self, root, split='train', classes=None, augment=False):
        split_dir = Path(root) / split
        if not split_dir.exists():
            raise FileNotFoundError(f'Missing split directory: {split_dir}')
        self.augment = bool(augment and split == 'train')
        if classes is None:
            classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir())
        self.classes = list(classes)
        paths, labels = [], []
        for idx, class_name in enumerate(self.classes):
            class_dir = split_dir / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f'Missing class folder: {class_dir}')
            for path in sorted(class_dir.glob('*.npy')):
                paths.append(path)
                labels.append(idx)
        self.clips = [normalize_clip(np.load(p)).astype(np.float16) for p in paths]
        self.labels = labels
        print(f'{split.upper()}: {len(self.clips)} samples loaded into memory', flush=True)

    def _horizontal_flip(self, sequence):
        if np.random.rand() < 0.5:
            sequence = np.flip(sequence, axis=2).copy()
        return sequence

    def _time_masking(self, sequence):
        if np.random.rand() < 0.5:
            length = sequence.shape[0]
            mask_len = np.random.randint(1, min(5, max(2, length // 6)))
            start = np.random.randint(0, length - mask_len + 1)
            sequence[start:start + mask_len] = sequence.mean(axis=0, keepdims=True)
        return sequence

    def __getitem__(self, index):
        sequence = self.clips[index].astype(np.float32)
        if self.augment:
            sequence = self._horizontal_flip(sequence)
            sequence = self._time_masking(sequence)
        return torch.from_numpy(sequence).unsqueeze(0), torch.tensor(self.labels[index], dtype=torch.long)

    def __len__(self):
        return len(self.clips)

## 5. Models
Every model takes `[B, 1, T, H, W]` and returns `[B, 23]` logits.
- Family A (models 1–5) shares `CNN3DBackbone`, so differences between them come only from the temporal back end.
- Family B uses the torchvision 3D ResNet-18 (R3D-18), trained from scratch.
- Family C uses an ImageNet-pretrained 2D ResNet-18 or ResNet-50 on each frame, with or without a temporal back end.

To add a model: write a class, add one line to `MODEL_REGISTRY`, and add its key to `MODELS_TO_RUN`.

In [ ]:
class CNN3DBackbone(nn.Module):
    # Shared spatiotemporal feature extractor (identical for every model).

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 32, (3, 5, 5), padding=(1, 2, 2)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),
            nn.Conv3d(32, 64, (3, 3, 3), padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),
            nn.Conv3d(64, 128, (3, 3, 3), padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)


def spatial_mean_sequence(features):
    # [B, C, T, H, W] -> [B, T, C]: average space only, keep temporal order.
    return features.mean(dim=(-1, -2)).transpose(1, 2)


class CNN3D(nn.Module):
    # Baseline: 3D-CNN followed by global adaptive average pooling.

    def __init__(self, num_classes):
        super().__init__()
        self.backbone = CNN3DBackbone()
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.head(self.pool(self.backbone(x)).flatten(1))


class CNN3DRNN(nn.Module):
    # 3D-CNN features followed by a recurrent temporal layer (last time step -> classifier).

    def __init__(self, num_classes, rnn_type, hidden_size=128, num_layers=1, dropout=0.0):
        super().__init__()
        if rnn_type not in {'lstm', 'gru'}:
            raise ValueError("rnn_type must be 'lstm' or 'gru'")
        self.backbone = CNN3DBackbone()
        rnn_cls = nn.LSTM if rnn_type == 'lstm' else nn.GRU
        self.rnn = rnn_cls(input_size=128, hidden_size=hidden_size, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        sequence, _ = self.rnn(spatial_mean_sequence(self.backbone(x)))
        return self.head(sequence[:, -1, :])


class CNN3DTemporalAttention(nn.Module):
    # 3D-CNN + temporal self-attention + attention pooling.
    # 1. Average over space -> one 128-d vector per time step (same input as LSTM/GRU).
    # 2. Learnable positional embedding, so the model knows frame order.
    # 3. One multi-head self-attention layer (4 heads), residual + LayerNorm.
    # 4. Attention pooling: a learned weight per time step instead of mean / last step.

    def __init__(self, num_classes, dim=128, heads=4, max_len=64):
        super().__init__()
        self.backbone = CNN3DBackbone()
        self.max_len = max_len
        self.position = nn.Parameter(torch.zeros(1, max_len, dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, dropout=0.0, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def temporal_features(self, x):
        h = spatial_mean_sequence(self.backbone(x))
        steps = h.size(1)
        if steps > self.max_len:
            raise ValueError(f'Sequence has {steps} steps after pooling; increase max_len.')
        h = h + self.position[:, :steps]
        attended, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + attended)
        weights = torch.softmax(self.score(h), dim=1)  # [B, T, 1]
        return h, weights

    def forward(self, x):
        h, weights = self.temporal_features(x)
        return self.head((weights * h).sum(dim=1))


class CNN3DSpatiotemporalAttention(nn.Module):
    # 3D-CNN + spatiotemporal self-attention + attention pooling.
    # Unlike the temporal model, space is NOT averaged away first. The CNN map is pooled
    # to a fixed 4x4 grid per time step, so every resolution gives the same number of
    # tokens (T' x 16, about 14 x 16 = 224). Each token gets a time + a space position
    # embedding, and one self-attention layer lets every (time, region) token attend to
    # every other one, e.g. lip corners in frame 3 to the mouth centre in frame 10.
    # Same dim / heads / pooling as the temporal model, so the only change is the tokens.

    def __init__(self, num_classes, dim=128, heads=4, grid=4, max_len=64):
        super().__init__()
        self.backbone = CNN3DBackbone()
        self.grid = grid
        self.max_len = max_len
        self.time_position = nn.Parameter(torch.zeros(1, max_len, 1, dim))
        self.space_position = nn.Parameter(torch.zeros(1, 1, grid * grid, dim))
        nn.init.trunc_normal_(self.time_position, std=0.02)
        nn.init.trunc_normal_(self.space_position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, dropout=0.0, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def tokens(self, x):
        f = self.backbone(x)                                    # [B, C, T, H, W]
        b, c, t = f.shape[:3]
        if t > self.max_len:
            raise ValueError(f'Sequence has {t} steps after pooling; increase max_len.')
        f = F.adaptive_avg_pool3d(f, (t, self.grid, self.grid))  # [B, C, T, g, g]
        h = f.flatten(3).permute(0, 2, 3, 1)                     # [B, T, g*g, C]
        h = h + self.time_position[:, :t] + self.space_position
        return h.reshape(b, t * self.grid * self.grid, c)       # [B, T*g*g, C]

    def forward(self, x):
        h = self.tokens(x)
        attended, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + attended)
        weights = torch.softmax(self.score(h), dim=1)            # [B, tokens, 1]
        return self.head((weights * h).sum(dim=1))


# ============================================================================
# FAMILY B: 3D ResNet-18 (torchvision R3D-18). These models do NOT use CNN3DBackbone.
# Standard torchvision networks, trained from scratch with exactly the same data,
# resolutions, seeds and training protocol.
# ============================================================================
from torchvision.models import resnet50
from torchvision.models.video import r3d_18


class VideoResNet(nn.Module):
    # Plain torchvision 3D ResNet-18 (R3D-18), random init, grayscale stem, 23-way head.
    def __init__(self, num_classes):
        super().__init__()
        net = r3d_18(weights=None)
        old = net.stem[0]
        net.stem[0] = nn.Conv3d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
        self.net = net

    def forward(self, x):          # x: [B, 1, T, H, W]
        return self.net(x)


# 3D ResNet-18 trunk + our heads. One change to R3D-18: layers 3 and 4 keep the time length
# (temporal stride 1 instead of 2), so the heads get 15 time steps instead of only 4.
class R3D18Trunk(nn.Module):
    # torchvision R3D-18 without its pooling/classifier, grayscale input. Output: [B, 512, 15, H/16, W/16].
    def __init__(self):
        super().__init__()
        net = r3d_18(weights=None)
        old = net.stem[0]
        net.stem[0] = nn.Conv3d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        for layer in (net.layer3, net.layer4):          # keep time: stride (2,2,2) -> (1,2,2)
            layer[0].conv1[0].stride = (1, 2, 2)
            layer[0].downsample[0].stride = (1, 2, 2)
        self.body = nn.Sequential(net.stem, net.layer1, net.layer2, net.layer3, net.layer4)

    def forward(self, x):
        return self.body(x)


class R3DLSTM(nn.Module):
    # R3D-18 trunk -> average over space -> 1-layer LSTM (256 units) -> last time step -> classifier.
    def __init__(self, num_classes, hidden=256):
        super().__init__()
        self.trunk = R3D18Trunk()
        self.rnn = nn.LSTM(512, hidden, batch_first=True)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        seq, _ = self.rnn(spatial_mean_sequence(self.trunk(x)))
        return self.head(seq[:, -1])


class R3DTemporalAttention(nn.Module):
    # R3D-18 trunk -> average over space -> position embedding -> 4-head self-attention over time
    # -> residual + LayerNorm -> attention pooling -> classifier (same design as our temporal model).
    def __init__(self, num_classes, dim=512, heads=4, max_len=64):
        super().__init__()
        self.trunk = R3D18Trunk()
        self.position = nn.Parameter(torch.zeros(1, max_len, dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        h = spatial_mean_sequence(self.trunk(x))
        h = h + self.position[:, :h.size(1)]
        a, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + a)
        w = torch.softmax(self.score(h), dim=1)
        return self.head((w * h).sum(dim=1))


class R3DSpatiotemporalAttention(nn.Module):
    # R3D-18 trunk -> pool to a fixed 2x2 grid per time step (15 x 4 = 60 tokens at every resolution)
    # -> time + space embeddings -> 4-head self-attention over all space-time tokens
    # -> residual + LayerNorm -> attention pooling -> classifier (same design as our spatiotemporal model).
    def __init__(self, num_classes, dim=512, heads=4, grid=2, max_len=64):
        super().__init__()
        self.trunk = R3D18Trunk()
        self.grid = grid
        self.time_position = nn.Parameter(torch.zeros(1, max_len, 1, dim))
        self.space_position = nn.Parameter(torch.zeros(1, 1, grid * grid, dim))
        nn.init.trunc_normal_(self.time_position, std=0.02)
        nn.init.trunc_normal_(self.space_position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        f = self.trunk(x)
        b, c, t = f.shape[:3]
        f = F.adaptive_avg_pool3d(f, (t, self.grid, self.grid))
        h = f.flatten(3).permute(0, 2, 3, 1) + self.time_position[:, :t] + self.space_position
        h = h.reshape(b, t * self.grid * self.grid, c)
        a, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + a)
        w = torch.softmax(self.score(h), dim=1)
        return self.head((w * h).sum(dim=1))


class ResNet50TemporalAttention(nn.Module):
    def __init__(self, num_classes, dim=512, heads=4, max_len=64):
        super().__init__()
        net = resnet50(weights=None)
        old = net.conv1
        net.conv1 = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        net.fc = nn.Identity()                       # keep the 2048-d pooled frame feature
        self.frames = net
        self.proj = nn.Linear(2048, dim)
        self.position = nn.Parameter(torch.zeros(1, max_len, dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):                            # x: [B, 1, T, H, W]
        b, c, t, hh, ww = x.shape
        f = self.frames(x.transpose(1, 2).reshape(b * t, c, hh, ww))     # [B*T, 2048]
        h = self.proj(f.view(b, t, -1)) + self.position[:, :t]            # [B, T, 512]
        a, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + a)
        w = torch.softmax(self.score(h), dim=1)
        return self.head((w * h).sum(dim=1))




# ============================================================================
# FAMILY C: PRETRAINED 2D FRAME ENCODERS + TEMPORAL BACK ENDS
#   1. TCN                       2. TCN + temporal attention
#   3. ResNet-18 + BiGRU + att.  4. ResNet-50 + BiGRU + att.
# The frame encoder is a 2D ResNet pretrained on ImageNet, applied to every frame.
# Grayscale input: the pretrained RGB conv1 weights are summed over the 3 colour
# channels (standard trick). The pretrained encoder is fine-tuned with a 10x smaller
# learning rate than the new layers (see param_groups + train()).
# ============================================================================
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights

USE_PRETRAINED = True     # False only for offline tests


class FrameEncoder(nn.Module):
    # 2D ResNet (ImageNet weights) on each frame -> [B, T, out_dim]
    def __init__(self, arch='resnet18', out_dim=256):
        super().__init__()
        if arch == 'resnet18':
            net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if USE_PRETRAINED else None); feat = 512
        else:
            net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if USE_PRETRAINED else None); feat = 2048
        w = net.conv1.weight.data.sum(dim=1, keepdim=True)          # RGB -> 1 channel
        net.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        net.conv1.weight.data.copy_(w)
        net.fc = nn.Identity()
        self.net = net
        self.proj = nn.Linear(feat, out_dim)

    def forward(self, x):                                          # x: [B, 1, T, H, W]
        b, c, t, h, w = x.shape
        f = self.net(x.transpose(1, 2).reshape(b * t, c, h, w))     # [B*T, feat]
        return self.proj(f.view(b, t, -1))                          # [B, T, out_dim]


class AttentionHead(nn.Module):
    # Same temporal attention as our best 3D-CNN model: position embedding, 4-head
    # self-attention, residual + LayerNorm, attention pooling, linear classifier.
    def __init__(self, dim, num_classes, heads=4, max_len=64):
        super().__init__()
        self.position = nn.Parameter(torch.zeros(1, max_len, dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, h):                                          # h: [B, T, dim]
        h = h + self.position[:, :h.size(1)]
        a, _ = self.attention(h, h, h, need_weights=False)
        h = self.norm(h + a)
        w = torch.softmax(self.score(h), dim=1)
        return self.head((w * h).sum(dim=1))


class TCNBlock(nn.Module):
    # Residual block of two dilated 1D convolutions over time (non-causal, same length).
    def __init__(self, ch, dilation, kernel=3, dropout=0.2):
        super().__init__()
        pad = dilation * (kernel - 1) // 2
        self.body = nn.Sequential(
            nn.Conv1d(ch, ch, kernel, padding=pad, dilation=dilation), nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(ch, ch, kernel, padding=pad, dilation=dilation), nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, x):                                          # x: [B, C, T]
        return torch.relu(x + self.body(x))


class TCN(nn.Module):
    # 4 residual blocks, dilations 1, 2, 4, 8 (receptive field covers all 29 frames).
    def __init__(self, ch=256):
        super().__init__()
        self.blocks = nn.Sequential(*[TCNBlock(ch, d) for d in (1, 2, 4, 8)])

    def forward(self, h):                                          # h: [B, T, C]
        return self.blocks(h.transpose(1, 2)).transpose(1, 2)


class PretrainedTemporalModel(nn.Module):
    # frame encoder (pretrained ResNet) -> back-end -> classifier
    #   backend 'tcn'      : TCN, then average over time
    #   backend 'tcn_att'  : TCN, then temporal attention head
    #   backend 'bigru_att': bidirectional GRU (2 x 128), then temporal attention head
    #   backend 'att'      : temporal attention head directly on the frame features
    def __init__(self, num_classes, arch='resnet18', backend='tcn', dim=256):
        super().__init__()
        self.encoder = FrameEncoder(arch, dim)
        self.backend = backend
        if backend in ('tcn', 'tcn_att'):
            self.temporal = TCN(dim)
        elif backend == 'bigru_att':
            self.temporal = nn.GRU(dim, dim // 2, batch_first=True, bidirectional=True)
        else:                                                      # 'att': attention directly on frame features
            self.temporal = nn.Identity()
        self.att = AttentionHead(dim, num_classes) if backend != 'tcn' else None
        self.fc = nn.Linear(dim, num_classes) if backend == 'tcn' else None

    def forward(self, x):
        h = self.encoder(x)
        h = self.temporal(h)[0] if self.backend == 'bigru_att' else self.temporal(h)   # 'att': identity
        return self.fc(h.mean(dim=1)) if self.backend == 'tcn' else self.att(h)

    def param_groups(self, lr):
        enc = list(self.encoder.net.parameters())
        ids = {id(p) for p in enc}
        rest = [p for p in self.parameters() if id(p) not in ids]
        return [{'params': enc, 'lr': lr * 0.1}, {'params': rest, 'lr': lr}]


class ResNetAlone(nn.Module):
    # Pretrained 2D ResNet per frame, no temporal model:
    # frame features -> average over time -> linear classifier (baseline for the ResNet family)
    def __init__(self, num_classes, arch='resnet50', dim=256):
        super().__init__()
        self.encoder = FrameEncoder(arch, dim)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, x):
        h = self.encoder(x)                      # [B, T, dim]
        return self.fc(h.mean(dim=1))

    def param_groups(self, lr):
        enc = list(self.encoder.net.parameters())
        ids = {id(p) for p in enc}
        rest = [p for p in self.parameters() if id(p) not in ids]
        return [{'params': enc, 'lr': lr * 0.1}, {'params': rest, 'lr': lr}]


# ---- Registry: key -> (label for tables/figures, builder) ------------------
MODEL_REGISTRY = {
    '3d_cnn':              ('3D-CNN (Baseline)',                  lambda: CNN3D(NUM_CLASSES)),
    '3d_cnn_lstm':         ('3D-CNN + LSTM',                      lambda: CNN3DRNN(NUM_CLASSES, 'lstm')),
    '3d_cnn_gru':          ('3D-CNN + GRU',                       lambda: CNN3DRNN(NUM_CLASSES, 'gru')),
    '3d_cnn_attention':    ('3D-CNN + Temporal Attention',        lambda: CNN3DTemporalAttention(NUM_CLASSES)),
    '3d_cnn_st_attention': ('3D-CNN + Spatiotemporal Attention',  lambda: CNN3DSpatiotemporalAttention(NUM_CLASSES)),
    # Family B
    'r3d_18':              ('3D ResNet-18',                       lambda: VideoResNet(NUM_CLASSES)),
    'r3d_attention':       ('3D ResNet-18 + Temporal Attention',  lambda: R3DTemporalAttention(NUM_CLASSES)),
    # ResNet-50 from scratch (preliminary test only, not reported: it did not learn)
    'r50_attention':       ('ResNet-50 + Temporal Attention',     lambda: ResNet50TemporalAttention(NUM_CLASSES)),
    # Family C
    'r18p_tcn':            ('ResNet-18 (pretrained) + TCN',                   lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet18', 'tcn')),
    'r18p_tcn_att':        ('ResNet-18 (pretrained) + TCN + Temporal Attention', lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet18', 'tcn_att')),
    'r18p_bigru_att':      ('ResNet-18 (pretrained) + BiGRU + Attention',     lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet18', 'bigru_att')),
    'r50p_bigru_att':      ('ResNet-50 (pretrained) + BiGRU + Attention',     lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet50', 'bigru_att')),
    'r50p_tcn':            ('ResNet-50 (pretrained) + TCN',                   lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet50', 'tcn')),
    'r50p_att':            ('ResNet-50 (pretrained) + Temporal Attention',    lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet50', 'att')),
    'r50p_tcn_att':        ('ResNet-50 (pretrained) + TCN + Temporal Attention', lambda: PretrainedTemporalModel(NUM_CLASSES, 'resnet50', 'tcn_att')),
    # Family C: no temporal model
    'r50p_avg':            ('ResNet-50 (pretrained)',                         lambda: ResNetAlone(NUM_CLASSES, 'resnet50')),
    'r18p_avg':            ('ResNet-18 (pretrained)',                         lambda: ResNetAlone(NUM_CLASSES, 'resnet18')),
    # optional extra heads on the 3D ResNet-18 (defined, not in MODELS_TO_RUN by default)
    'r3d_lstm':            ('3D ResNet-18 + LSTM',                lambda: R3DLSTM(NUM_CLASSES)),
    'r3d_st_attention':    ('3D ResNet-18 + Spatiotemporal Attention', lambda: R3DSpatiotemporalAttention(NUM_CLASSES)),
}
MODEL_LABELS = {k: v[0] for k, v in MODEL_REGISTRY.items()}
BASELINE = '3d_cnn'


def build_model(name):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown model {name!r}. Known: {list(MODEL_REGISTRY)}')
    return MODEL_REGISTRY[name][1]()


def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 6. Smoke test (seconds, no data needed)
Builds every model, runs random clips at the largest and smallest resolution, and checks the output shape and parameter counts.

In [ ]:
EXPECTED_PARAMS = {'3d_cnn': 282519, '3d_cnn_lstm': 414615, '3d_cnn_gru': 381591,
                   '3d_cnn_attention': 357144, '3d_cnn_st_attention': 359192}
torch.manual_seed(11)
for name in MODEL_REGISTRY:
    model = build_model(name).eval()
    with torch.no_grad():
        for res in (max(RESOLUTIONS), min(RESOLUTIONS)):
            out = model(torch.randn(2, 1, 29, res, res))   # 29 frames, like the real clips
            assert tuple(out.shape) == (2, NUM_CLASSES), f'{name} @ {res}: got {tuple(out.shape)}'
    n = parameter_count(model)
    if name in EXPECTED_PARAMS:
        assert n == EXPECTED_PARAMS[name], f'{name}: {n} params, expected {EXPECTED_PARAMS[name]}'
    print(f'{MODEL_LABELS[name]:<36} OK   trainable parameters = {n:,}')
print('Smoke test passed.')

## 7. Training and evaluation

In [ ]:
FIELDS = ['model', 'model_label', 'seed', 'resolution', 'n_test', 'accuracy', 'precision',
          'recall', 'f1', 'best_epoch', 'seconds']


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resize_batch(x, resolution):
    if x.shape[-2:] == (resolution, resolution):
        return x
    b, c, t, h, w = x.shape
    y = x.permute(0, 2, 1, 3, 4).reshape(b * t, c, h, w)
    y = F.interpolate(y, size=(resolution, resolution), mode='bilinear',
                      align_corners=False, antialias=True)
    return y.reshape(b, t, c, resolution, resolution).permute(0, 2, 1, 3, 4)


def metrics_from_predictions(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(y_true, y_pred) * 100), 'precision': float(p * 100),
            'recall': float(r * 100), 'f1': float(f * 100)}


def evaluate(model, loader, resolution):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(resize_batch(x.to(DEVICE), resolution))
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())
            y_true.extend(y.tolist())
    return metrics_from_predictions(y_true, y_pred), np.asarray(y_true), np.asarray(y_pred)


def train(model, loader, val_loader, resolution, epochs, learning_rate, verbose=False):
    criterion = nn.CrossEntropyLoss()
    # pretrained models fine-tune their encoder with a smaller LR (param_groups); all others: one LR
    base = getattr(model, 'module', model)          # unwrap DataParallel
    groups = base.param_groups(learning_rate) if hasattr(base, 'param_groups') else \
        [{'params': list(model.parameters()), 'lr': learning_rate}]
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[g['lr'] for g in groups], total_steps=epochs * len(loader), pct_start=0.15)
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best_score, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = total = correct = 0
        for x, y in loader:
            x = resize_batch(x.to(DEVICE, non_blocking=True), resolution)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
        val_metrics, _, _ = evaluate(model, val_loader, resolution)
        history.append({'epoch': epoch, 'train_loss': total_loss / max(total, 1),
                        'train_accuracy': 100 * correct / max(total, 1),
                        'validation_accuracy': val_metrics['accuracy'],
                        'validation_f1': val_metrics['f1']})
        if verbose:
            h = history[-1]
            print(f"    epoch {epoch:2d}  loss {h['train_loss']:.3f}  train {h['train_accuracy']:5.1f}%  "
                  f"val {h['validation_accuracy']:5.1f}%", flush=True)
        if history[-1]['validation_accuracy'] > best_score:
            best_score = history[-1]['validation_accuracy']
            best_state = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}
    base.load_state_dict(best_state)
    best_epoch = max(history, key=lambda h: h['validation_accuracy'])['epoch']
    return history, best_epoch, best_state


def finished_runs(out):
    csv_path = Path(out) / 'results_per_run.csv'
    done = set()
    if csv_path.exists():
        with open(csv_path, newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                done.add((r['model'], int(r['seed']), int(r['resolution'])))
    return done


def run_experiments(out, models, seeds, resolutions, epochs=EPOCHS, learning_rate=LEARNING_RATE,
                    verbose=False):
    # Trains every missing (seed, model, resolution) run. Saved after EVERY run:
    #   results_per_run.csv, predictions/*.npz, histories/*.json, checkpoints/*.pth
    out = Path(out)
    for folder in ('predictions', 'histories', 'checkpoints'):
        (out / folder).mkdir(parents=True, exist_ok=True)
    csv_path = out / 'results_per_run.csv'
    if not csv_path.exists():
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()
    done = finished_runs(out)
    # model by model: one model finishes all its seeds before the next one starts
    jobs = [(s, m, r) for m in models for s in seeds for r in resolutions if (m, s, r) not in done]
    print(f'Already finished: {len(done)} | runs to do now: {len(jobs)}', flush=True)
    for i, (seed, model_name, resolution) in enumerate(jobs, 1):
        hours = (time.time() - SESSION_START) / 3600
        if hours > TIME_BUDGET_HOURS:
            print(f'Time budget reached ({hours:.1f} h of {TIME_BUDGET_HOURS} h). Stopping here; '
                  f'{len(jobs) - i + 1} runs left for the next session.', flush=True)
            break
        seed_everything(seed)
        started = time.time()
        model = build_model(model_name).to(DEVICE)
        if USE_BOTH_GPUS and torch.cuda.device_count() > 1:   # Kaggle 'GPU T4 x2': split each batch over 2 GPUs
            model = nn.DataParallel(model)
        history, best_epoch, state = train(model, train_loader, val_loader, resolution,
                                           epochs, learning_rate, verbose)
        metrics, y_true, y_pred = evaluate(model, test_loader, resolution)
        name = f'{model_name}_seed{seed}_res{resolution}'
        np.savez(out / 'predictions' / f'{name}.npz', y_true=y_true, y_pred=y_pred)
        with open(out / 'histories' / f'{name}.json', 'w') as f:
            json.dump(history, f, indent=1)
        torch.save({'state_dict': state, 'classes': CLASSES}, out / 'checkpoints' / f'{name}.pth')
        row = {'model': model_name, 'model_label': MODEL_LABELS[model_name], 'seed': seed,
               'resolution': resolution, 'n_test': len(test_ds), **metrics,
               'best_epoch': best_epoch, 'seconds': round(time.time() - started, 1)}
        if csv_path.exists() and csv_path.stat().st_size > 0:           # make sure the file ends with a newline
            with open(csv_path, 'rb') as f:
                f.seek(-1, 2)
                needs_newline = f.read(1) not in (b'\n', b'\r')
            if needs_newline:
                with open(csv_path, 'a', encoding='utf-8') as f:
                    f.write('\n')
        with open(csv_path, 'a', newline='', encoding='utf-8') as f:   # written last = run complete
            csv.DictWriter(f, fieldnames=FIELDS).writerow(row)
        print(f"[{i}/{len(jobs)}] {MODEL_LABELS[model_name]} | seed {seed} | {resolution}x{resolution} | "
              f"acc {metrics['accuracy']:.1f} | F1 {metrics['f1']:.1f} | best epoch {best_epoch} | "
              f"{row['seconds']:.0f}s | predicts {len(set(y_pred.tolist()))}/{NUM_CLASSES} words",
              flush=True)
    print('Done. Run Step 12 for tables and figures.')

## 8. Load the data into memory (about 1–2 minutes, about 3.5 GB RAM)

In [ ]:
train_ds = PreprocessedLRWARDataset(DATA, 'train', CLASSES, augment=True)
val_ds = PreprocessedLRWARDataset(DATA, 'val', CLASSES, augment=False)
test_ds = PreprocessedLRWARDataset(DATA, 'test', CLASSES, augment=False)
pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
assert (len(train_ds), len(val_ds), len(test_ds)) == (3672, 460, 458), 'Unexpected split sizes'

## 9. What is already finished?

In [ ]:
def status_table(out=RESULTS, models=None, seeds=SEEDS, resolutions=RESOLUTIONS):
    models = models or list(MODEL_REGISTRY)
    done = finished_runs(out)
    rows = []
    for m in models:
        row = {'Model': MODEL_LABELS[m]}
        for s in seeds:
            n = sum((m, s, r) in done for r in resolutions)
            row[f'seed {s}'] = f'{n}/{len(resolutions)}'
        rows.append(row)
    total = len(models) * len(seeds) * len(resolutions)
    have = sum((m, s, r) in done for m in models for s in seeds for r in resolutions)
    print(f'Finished {have} / {total} runs in {out}')
    return pd.DataFrame(rows)

status_table()

## 10. Pilot: test one model quickly (optional)
Trains one model at one resolution in a separate folder (`pilot/`), so nothing here is mixed with the main results.

In [ ]:
PILOT_MODELS = []      # empty = skip; e.g. ['r50p_tcn_att'] to test one model
PILOT_RESOLUTION = 48
PILOT_DIR = WORK / 'pilot'

run_experiments(PILOT_DIR, PILOT_MODELS, seeds=[11], resolutions=[PILOT_RESOLUTION], verbose=True)

## 11. Full experiment (resumable)
Runs every missing combination of `MODELS_TO_RUN` × `SEEDS` × `RESOLUTIONS`. Each finished run is saved immediately, and finished runs are skipped when the notebook is run again.
On one NVIDIA T4, one run takes about 4–57 minutes depending on the model and resolution.

In [ ]:
run_experiments(RESULTS, MODELS_TO_RUN, SEEDS, RESOLUTIONS)

## 12. Tables and figures
Everything is built from the saved runs only. Files are saved in `results/analysis/`.
- `summary_table.csv` / `.md`: mean ± sample std over seeds
- `paired_vs_baseline.csv`: same seed and resolution, model minus baseline
- `per_class_accuracy.csv`, `fig_comparison_*.png`, `fig_curves_*.png`, `fig_confusion_*.png`

In [ ]:
METRICS = ['accuracy', 'precision', 'recall', 'f1']
METRIC_LABELS = {'accuracy': 'Accuracy (%)', 'precision': 'Macro precision (%)',
                 'recall': 'Macro recall (%)', 'f1': 'Macro F1-score (%)'}
# Categorical palette in fixed order + a distinct marker/line style per model,
# so the figures still read in grayscale print and for colour-blind readers.
COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948', '#7f5a3c']
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h']
LINESTYLES = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1)), '-', '--', '-.']

ANALYSIS = RESULTS / 'analysis'
ANALYSIS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RESULTS / 'results_per_run.csv').drop_duplicates(['model', 'seed', 'resolution'], keep='last')
models = [m for m in MODEL_REGISTRY if m in set(df['model'])]
res_sorted = sorted(df['resolution'].unique(), reverse=True)
style = {m: (COLORS[i % len(COLORS)], MARKERS[i % len(MARKERS)], LINESTYLES[i % len(LINESTYLES)])
         for i, m in enumerate(MODEL_REGISTRY)}

# --- Summary table (mean ± sample std) --------------------------------------
rows = []
for r in res_sorted:
    for m in models:
        sel = df[(df['model'] == m) & (df['resolution'] == r)]
        if sel.empty:
            continue
        row = {'Resolution': f'{r}×{r}', 'Model': MODEL_LABELS[m], 'Seeds': len(sel)}
        for k in METRICS:
            sd = sel[k].std(ddof=1) if len(sel) > 1 else float('nan')
            row[METRIC_LABELS[k]] = f'{sel[k].mean():.1f} ± {sd:.1f}' if len(sel) > 1 else f'{sel[k].mean():.1f}'
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(ANALYSIS / 'summary_table.csv', index=False, encoding='utf-8-sig')
try:   # Markdown copy for the paper draft (needs the small 'tabulate' package)
    (ANALYSIS / 'summary_table.md').write_text(summary.to_markdown(index=False), encoding='utf-8')
except ImportError:
    pass
if (summary['Seeds'] < len(SEEDS)).any():
    print(f'NOTE: some rows have fewer than {len(SEEDS)} seeds. Do not put them in the paper as final.')
print('Table: mean ± sample std over seeds')
display(summary)

# --- Paired comparison vs. baseline ----------------------------------------
key = df.set_index(['model', 'seed', 'resolution'])
paired = []
for r in res_sorted:
    for m in models:
        if m == BASELINE:
            continue
        for k in ['accuracy', 'f1']:
            d = np.array([key.loc[(m, s, r), k] - key.loc[(BASELINE, s, r), k] for s in SEEDS
                          if (m, s, r) in key.index and (BASELINE, s, r) in key.index])
            if len(d):
                paired.append({'Resolution': f'{r}×{r}', 'Model': MODEL_LABELS[m], 'Metric': k,
                               'Mean diff': round(d.mean(), 2),
                               'Std diff': round(d.std(ddof=1), 2) if len(d) > 1 else None,
                               'Seeds better / total': f'{(d > 0).sum()} / {len(d)}',
                               'Per-seed diffs': ' '.join(f'{v:+.1f}' for v in d)})
if paired:
    paired_df = pd.DataFrame(paired)
    paired_df.to_csv(ANALYSIS / 'paired_vs_baseline.csv', index=False, encoding='utf-8-sig')
    print('\nPaired comparison vs. baseline (same seed, same resolution)')
    display(paired_df[paired_df['Metric'] == 'accuracy'])

# --- Comparison figures -----------------------------------------------------
x = np.arange(len(res_sorted), dtype=float)
offsets = np.linspace(-0.27, 0.27, len(models)) if len(models) > 1 else [0.0]
for k in METRICS:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for off, m in zip(offsets, models):
        pts = []
        for i, r in enumerate(res_sorted):
            v = df[(df['model'] == m) & (df['resolution'] == r)][k]
            if len(v):
                pts.append((i + off, v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0))
        if pts:
            xi, mu, sd = zip(*pts)
            color, marker, ls = style[m]
            ax.errorbar(xi, mu, yerr=sd, color=color, marker=marker, markersize=7, linestyle=ls,
                        linewidth=2, capsize=4, label=MODEL_LABELS[m])
    ax.set_xticks(x, [f'{r}×{r}' for r in res_sorted])
    ax.set_xlabel('Mouth-region spatial resolution (pixels)')
    ax.set_ylabel(METRIC_LABELS[k])
    ax.grid(True, axis='y', alpha=0.25)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0))   # outside, never covers data
    fig.tight_layout()
    fig.savefig(ANALYSIS / f'fig_comparison_{k}.png', dpi=300, bbox_inches='tight')
    if k == 'accuracy':
        plt.show()
    plt.close(fig)

# --- Curves, confusion matrices, per-class accuracy --------------------------
top, first = max(res_sorted), min(SEEDS)
per_class = {}
for m in models:
    h_path = RESULTS / 'histories' / f'{m}_seed{first}_res{top}.json'
    if h_path.exists():
        h = json.loads(h_path.read_text())
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot([e['epoch'] for e in h], [e['train_accuracy'] for e in h], 'o-', color=COLORS[0],
                linewidth=2, markersize=5, label='Training accuracy')
        ax.plot([e['epoch'] for e in h], [e['validation_accuracy'] for e in h], 's--', color=COLORS[1],
                linewidth=2, markersize=5, label='Validation accuracy')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
        ax.set_title(f'{MODEL_LABELS[m]} ({top}×{top}, seed {first})')
        ax.grid(True, axis='y', alpha=0.25); ax.legend(frameon=False); fig.tight_layout()
        fig.savefig(ANALYSIS / f'fig_curves_{m}.png', dpi=300); plt.close(fig)
    p_path = RESULTS / 'predictions' / f'{m}_seed{first}_res{top}.npz'
    if p_path.exists():
        d = np.load(p_path)
        cm = confusion_matrix(d['y_true'], d['y_pred'], labels=range(NUM_CLASSES))
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(cm, cmap='Blues')
        t = np.arange(NUM_CLASSES)
        ax.set_xticks(t, [str(i + 1) for i in t], fontsize=7)
        ax.set_yticks(t, [str(i + 1) for i in t], fontsize=7)
        ax.set_xlabel('Predicted class ID'); ax.set_ylabel('True class ID')
        ax.set_title(f'{MODEL_LABELS[m]} ({top}×{top}, seed {first})')
        fig.colorbar(im, ax=ax, label='Number of clips'); fig.tight_layout()
        fig.savefig(ANALYSIS / f'fig_confusion_{m}.png', dpi=300); plt.close(fig)
    for r in res_sorted:
        yt, yp = [], []
        for s in SEEDS:
            p = RESULTS / 'predictions' / f'{m}_seed{s}_res{r}.npz'
            if p.exists():
                d = np.load(p); yt += d['y_true'].tolist(); yp += d['y_pred'].tolist()
        if yt:
            yt, yp = np.array(yt), np.array(yp)
            per_class[f'{MODEL_LABELS[m]} {r}×{r}'] = [
                round((yp[yt == c] == c).mean() * 100, 1) if (yt == c).any() else None
                for c in range(NUM_CLASSES)]
if per_class:
    pd.DataFrame(per_class, index=[f'{i + 1}. {w}' for i, w in enumerate(CLASSES)]) \
        .to_csv(ANALYSIS / 'per_class_accuracy.csv', encoding='utf-8-sig')
print('All tables and figures saved in:', ANALYSIS)

## 13. Baseline check
Compares training and validation accuracy of the temporal-averaging baseline to check whether its low accuracy comes from underfitting or overfitting.

In [ ]:
for res in RESOLUTIONS:
    rows = []
    for m in MODEL_REGISTRY:
        p = RESULTS / 'histories' / f'{m}_seed{min(SEEDS)}_res{res}.json'
        if p.exists():
            h = json.loads(p.read_text())
            best = max(h, key=lambda e: e['validation_accuracy'])
            rows.append({'Model': MODEL_LABELS[m], 'Final train acc': round(h[-1]['train_accuracy'], 1),
                         'Best val acc': round(best['validation_accuracy'], 1), 'Best epoch': best['epoch']})
    if rows:
        print(f'\n{res}×{res}, seed {min(SEEDS)}')
        display(pd.DataFrame(rows))

## 14. Download results

In [ ]:
import shutil
zip_base = WORK / 'RES200_results_v2'
tmp = WORK / '_export'
if tmp.exists():
    shutil.rmtree(tmp)
shutil.copytree(RESULTS, tmp / RESULTS.name, ignore=shutil.ignore_patterns('checkpoints'))
zip_path = shutil.make_archive(str(zip_base), 'zip', tmp)
shutil.rmtree(tmp)
print('Saved:', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)